# Local-to-local translation (L2L)

## Purpose

L2L shifts an incoming local expansion from a parent target centre to a child
target centre. It is the algebraic operation used in a downward FMM pass. This
notebook compares the shifted expansion with an independently constructed M2L
expansion at the child centre and evaluates both near that child.

## Mathematical definition

With $d=c_{\mathrm{child}}-c_{\mathrm{parent}}$,

$$L^{\mathrm{child}}_\beta =
\sum_{|\beta+\gamma|\le p}
\frac{d^\gamma}{\gamma!}L^{\mathrm{parent}}_{\beta+\gamma}.$$

Translation preserves the represented Taylor polynomial, but independently
truncated M2L expansions at two centres need not have identical coefficients.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
expansion_order = 5
n_sources = 70
random_seed = 42
source_centre = np.zeros(3)
parent_target_centre = np.array([4.0, 0.5, -0.25])
child_target_centre = np.array([4.15, 0.40, -0.15])
target_region_half_width = 0.08

## Operator chain

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = rng.uniform(-0.4, 0.4, size=(n_sources, 3))
dipole_moments = rng.normal(size=(n_sources, 3))

source_multipole = cdfmm.p2m_dipole(
    source_centre,
    source_positions,
    dipole_moments,
    order=expansion_order,
)
parent_local = cdfmm.m2l(
    source_multipole,
    source_centre,
    parent_target_centre,
    order=expansion_order,
)
translated_child_local = cdfmm.l2l(
    parent_local,
    parent_target_centre,
    child_target_centre,
    order=expansion_order,
)
direct_child_local = cdfmm.m2l(
    source_multipole,
    source_centre,
    child_target_centre,
    order=expansion_order,
)

print(f"Parent-child centre shift: {np.linalg.norm(child_target_centre - parent_target_centre):.3f}")
print(f"Relative coefficient difference: {np.linalg.norm(translated_child_local - direct_child_local) / np.linalg.norm(direct_child_local):.3e}")

## Evaluation near the child centre

In [ ]:
target_offsets = rng.uniform(
    -target_region_half_width,
    target_region_half_width,
    size=(24, 3),
)
target_positions = child_target_centre + target_offsets
reference_fields = direct_fields(target_positions, source_positions, dipole_moments)
translated_fields = local_fields(
    target_positions,
    translated_child_local,
    child_target_centre,
    expansion_order,
)
direct_child_fields = local_fields(
    target_positions,
    direct_child_local,
    child_target_centre,
    expansion_order,
)

print("Translated L2L metrics:", error_metrics(translated_fields, reference_fields))
print("Direct child M2L metrics:", error_metrics(direct_child_fields, reference_fields))

## Geometry and pointwise errors

In [ ]:
figure = plt.figure(figsize=(13, 5))
geometry_axes = figure.add_subplot(121, projection="3d")
error_axes = figure.add_subplot(122)

geometry_axes.scatter(*source_positions.T, s=10, color="tab:blue", label="sources")
geometry_axes.scatter(*parent_target_centre, marker="X", s=100, color="tab:orange", label="parent local centre")
geometry_axes.scatter(*child_target_centre, marker="X", s=100, color="tab:green", label="child local centre")
geometry_axes.scatter(*target_positions.T, s=18, color="tab:red", label="evaluation targets")
finish_3d_axes(geometry_axes, "L2L geometry")
geometry_axes.legend(fontsize=8)

translated_error = relative_error(translated_fields, reference_fields)
direct_child_error = relative_error(direct_child_fields, reference_fields)
error_axes.semilogy(translated_error, "o-", label="M2L(parent) + L2L")
error_axes.semilogy(direct_child_error, "s-", label="M2L(child)")
error_axes.set_xlabel("Target index")
error_axes.set_ylabel("Relative field error")
error_axes.set_title("Local-expansion accuracy near child centre")
error_axes.legend()
figure.tight_layout()

## What to observe

The two child-centred coefficient vectors can differ because M2L and L2L
truncate at different stages. Their evaluated fields should nevertheless be
close in the small child region and converge as order increases or the
parent-child shift decreases.